# Random Forest — klasyfikacja krok po kroku

**Cel szkoleniowy:** pokazać, jak działa las losowy, jak oceniać model klasyfikacyjny oraz jak interpretować ważność cech.

W notebooku:
1. wczytamy dane,
2. zbudujemy model bazowy,
3. ocenimy jakość,
4. pokażemy macierz pomyłek i ROC,
5. sprawdzimy ważność cech,
6. zmienimy najważniejsze hiperparametry,
7. wykonamy prosty eksperyment porównawczy.

## 1. Intuicja

Random Forest buduje **wiele drzew decyzyjnych** na losowych próbkach danych i losowych podzbiorach cech.

Dla klasyfikacji:
- każde drzewo głosuje na klasę,
- wynik końcowy to głos większości.

Najważniejsze hiperparametry:
- `n_estimators` — liczba drzew,
- `max_depth` — maksymalna głębokość drzewa,
- `min_samples_split` — minimalna liczba próbek wymagana do podziału,
- `min_samples_leaf` — minimalna liczba próbek w liściu,
- `max_features` — ile cech losujemy przy podziale,
- `class_weight` — sposób uwzględniania niezbalansowanych klas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)

RANDOM_STATE = 42

## 2. Dane

Użyjemy wbudowanego zbioru **Breast Cancer Wisconsin** z `scikit-learn`.

To klasyfikacja binarna:
- 0 — nowotwór złośliwy,
- 1 — nowotwór łagodny.

Zbiór ma 30 cech liczbowych opisujących właściwości jąder komórkowych.

In [ ]:
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("Rozmiar X:", X.shape)
print("\nRozkład klas:")
print(y.value_counts().rename(index=dict(enumerate(data.target_names))))

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

## 3. Model bazowy

Na początek używamy 200 drzew i pozostawiamy większość ustawień domyślnych.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC AUC :", round(roc_auc_score(y_test, y_proba), 4))
print("\nRaport klasyfikacji:\n")
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. Macierz pomyłek

Macierz pomyłek odpowiada na pytanie: **gdzie model się myli?**

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=data.target_names
)
plt.title("Random Forest — macierz pomyłek")
plt.show()

## 5. Krzywa ROC

Krzywa ROC pokazuje kompromis między:
- **TPR / recall**,
- **FPR**.

Im większe pole AUC, tym lepiej model rozdziela klasy.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Random Forest — ROC")
plt.show()

## 6. Ważność cech

Random Forest potrafi oszacować, które cechy były najbardziej użyteczne przy podejmowaniu decyzji.

In [ ]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

top15 = importance.head(15).sort_values()

plt.figure(figsize=(9, 6))
top15.plot(kind="barh")
plt.title("15 najważniejszych cech")
plt.xlabel("Feature importance")
plt.show()

importance.head(10)

## 7. Eksperyment: liczba drzew i głębokość

Sprawdzimy kilka konfiguracji.

**Pytanie do grupy:**  
Czy więcej drzew zawsze daje wyraźnie lepszy wynik?

In [ ]:
results = []

for n_estimators in [20, 50, 100, 300]:
    for max_depth in [None, 3, 5, 8]:
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        results.append({
            "n_estimators": n_estimators,
            "max_depth": str(max_depth),
            "accuracy": accuracy_score(y_test, pred)
        })

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df.head(10)

In [ ]:
pivot = results_df.pivot(
    index="max_depth",
    columns="n_estimators",
    values="accuracy"
)

pivot

## 8. Opcje do pokazania na szkoleniu

Można zmieniać:
- `n_estimators=50 / 200 / 500`,
- `max_depth=None / 3 / 5 / 10`,
- `max_features="sqrt" / "log2"`,
- `min_samples_leaf=1 / 3 / 5`,
- `class_weight=None / "balanced"`.

### Ćwiczenie
1. Ustaw `max_depth=3`.
2. Porównaj accuracy z modelem bez ograniczenia głębokości.
3. Zmień `n_estimators` na 20 i 500.
4. Sprawdź, czy ranking najważniejszych cech się zmienił.

### Wniosek
Random Forest jest dobrym modelem bazowym, bo:
- wymaga niewielkiego przygotowania danych,
- jest odporny na nieliniowości,
- dobrze radzi sobie z wieloma cechami,
- daje możliwość interpretacji przez ważność cech.